# Bias, Varianza y Ruido: Por que los modelos no son perfectos

En Machine Learning, **ningun modelo logra error cero**. El error total de un modelo se puede descomponer en tres componentes fundamentales:

$$
\text{Error Total} = \text{Bias}^2 + \text{Varianza} + \text{Ruido Irreducible}
$$

Cada componente tiene un origen distinto:

| Componente | Origen | Pregunta clave |
|---|---|---|
| **Bias (Sesgo)** | El modelo es demasiado simple para capturar la relacion real entre X e Y. | "Esta el modelo sistematicamente equivocado?" |
| **Varianza** | El modelo es demasiado sensible a los datos de entrenamiento especificos. | "Cuanto cambian las predicciones si cambio el dataset?" |
| **Ruido Irreducible** | Aleatoriedad intrinseca en los datos que ningun modelo puede eliminar. | "Existe informacion que simplemente no esta disponible?" |

Esta descomposicion es la clave para entender **por que un modelo falla** y **que estrategia usar para mejorarlo**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Estilo de graficos
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

SEED = 42
np.random.seed(SEED)
print("Librerias cargadas correctamente.")

## Las tres fuentes de error en detalle

### 1. Bias (Sesgo): el modelo no puede aprender la relacion real

El bias mide **que tan lejos esta, en promedio, la prediccion del modelo respecto al valor verdadero**. Un modelo con alto bias es demasiado simple: asume una forma funcional que no corresponde con la realidad.

- **Ejemplo**: ajustar una linea recta a datos que siguen una curva sinusoidal.
- **Diagnostico**: error alto tanto en entrenamiento como en test (underfitting).
- **Solucion**: aumentar la complejidad del modelo, agregar features, usar modelos no lineales.

### 2. Varianza: el modelo memoriza el ruido

La varianza mide **cuanto cambian las predicciones cuando se entrena con diferentes muestras del mismo problema**. Un modelo con alta varianza se adapta excesivamente a los datos de entrenamiento, capturando patrones que son solo ruido.

- **Ejemplo**: un polinomio de grado 15 que pasa exactamente por todos los puntos de entrenamiento.
- **Diagnostico**: error bajo en entrenamiento pero alto en test (overfitting).
- **Solucion**: regularizacion, mas datos, reducir complejidad, ensemble methods.

### 3. Ruido Irreducible: informacion que no esta en los datos

El ruido irreducible representa la **aleatoriedad intrinseca del fenomeno**. Incluso si tuvieramos el modelo perfecto (la funcion verdadera f(x)), seguiriamos teniendo error porque existe variabilidad que no puede ser explicada por las variables disponibles.

- **Ejemplo**: predecir la duracion exacta de un vuelo sin conocer las condiciones meteorologicas.
- **Diagnostico**: existe un piso de error que ningun modelo puede superar.
- **Solucion**: no hay solucion desde el modelado. Solo se puede reducir recopilando mas variables informativas.

In [ ]:
# --- Demostracion: Underfitting, Optimo y Overfitting ---

np.random.seed(SEED)

# Datos: seno con ruido
n = 50
X = np.sort(np.random.uniform(0, 2 * np.pi, n))
noise = np.random.normal(0, 0.3, n)
y_true = np.sin(X)
y = y_true + noise

# Linea suave para graficar la funcion verdadera y las predicciones
X_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

# Modelos con distintos grados de complejidad
degrees = [1, 4, 15]
labels = [
    'Grado 1 (alto bias, underfitting)',
    'Grado 4 (equilibrio optimo)',
    'Grado 15 (alta varianza, overfitting)'
]
colors = ['#e74c3c', '#27ae60', '#2980b9']
linestyles = ['--', '-', '-.']

fig, ax = plt.subplots(figsize=(12, 7))

# Datos observados
ax.scatter(X, y, color='gray', alpha=0.6, s=40, label='Datos observados (y = sin(x) + ruido)', zorder=5)

# Funcion verdadera
ax.plot(X_plot, np.sin(X_plot), 'k-', linewidth=2, alpha=0.4, label='Funcion verdadera: f(x) = sin(x)')

# Ajustar y graficar cada modelo
for deg, label, color, ls in zip(degrees, labels, colors, linestyles):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X.reshape(-1, 1), y)
    y_pred = model.predict(X_plot)
    ax.plot(X_plot, y_pred, color=color, linewidth=2, linestyle=ls, label=label)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Efecto de la complejidad del modelo sobre el ajuste')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

## El tradeoff Bias-Varianza

No se pueden minimizar bias y varianza simultaneamente. Existe un **compromiso** (tradeoff):

```
  Error
    |
    |  \                          /
    |   \   Bias^2              /  Varianza
    |    \                    /
    |     \       __________/
    |      \_____/
    |   .....................  <-- Ruido irreducible (constante)
    |
    +------------------------------------>
     Simple                    Complejo
              Complejidad del modelo
```

- **Modelos simples** (pocos parametros): alto bias, baja varianza. No capturan la senal.
- **Modelos complejos** (muchos parametros): bajo bias, alta varianza. Capturan senal y ruido.
- **Punto optimo**: la complejidad que minimiza el **error total** (la suma de las tres componentes).

El grafico siguiente muestra este fenomeno empiricamente con las curvas de error en train y test.

In [ ]:
# --- Curvas de error Train vs Test por complejidad ---

np.random.seed(SEED)

n = 50
X = np.sort(np.random.uniform(0, 2 * np.pi, n))
noise = np.random.normal(0, 0.3, n)
y = np.sin(X) + noise

X_train, X_test, y_train, y_test = train_test_split(
    X.reshape(-1, 1), y, test_size=0.3, random_state=SEED
)

max_degree = 15
degrees = range(1, max_degree + 1)
train_errors = []
test_errors = []

for deg in degrees:
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_train, y_train)
    train_errors.append(mean_squared_error(y_train, model.predict(X_train)))
    test_errors.append(mean_squared_error(y_test, model.predict(X_test)))

# Encontrar el grado optimo (minimo error en test)
optimal_deg = list(degrees)[np.argmin(test_errors)]
optimal_test_error = min(test_errors)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(degrees), train_errors, 'o-', color='#2980b9', linewidth=2, label='Error en Train')
ax.plot(list(degrees), test_errors, 's-', color='#e74c3c', linewidth=2, label='Error en Test')
ax.axvline(optimal_deg, color='#27ae60', linestyle='--', alpha=0.7,
           label=f'Grado optimo = {optimal_deg}')
ax.scatter([optimal_deg], [optimal_test_error], color='#27ae60', s=150, zorder=5,
           edgecolors='black', linewidths=1.5)
ax.annotate(f'Minimo test error\nGrado={optimal_deg}, MSE={optimal_test_error:.4f}',
            xy=(optimal_deg, optimal_test_error),
            xytext=(optimal_deg + 2, optimal_test_error + 0.1),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, ha='left')

ax.set_xlabel('Grado del polinomio (complejidad del modelo)')
ax.set_ylabel('Error Cuadratico Medio (MSE)')
ax.set_title('Curva U del error en Test: evidencia del tradeoff Bias-Varianza')
ax.set_xticks(list(degrees))
ax.legend(fontsize=11)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f"Grado optimo: {optimal_deg}")
print(f"MSE en test del modelo optimo: {optimal_test_error:.4f}")

## Error de Bayes y el ruido irreducible

El **Error de Bayes** (tambien llamado error irreducible) es el **piso minimo de error** que cualquier modelo puede alcanzar, incluso si conocieramos la funcion verdadera $f(x)$ perfectamente.

$$
\text{Error de Bayes} = E[\varepsilon^2] = \sigma^2_{\text{ruido}}
$$

Esto ocurre porque los datos observados son:

$$
y = f(x) + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)
$$

Incluso el **oraculo perfecto** (el modelo que conoce $f(x)$ exactamente) tiene un error residual igual a $\sigma^2$. Este error proviene de variables latentes, mediciones imprecisas, o simple aleatoriedad del fenomeno.

**Implicacion practica**: cuando el error de tu modelo en test se acerca al nivel de ruido estimado, **ya no tiene sentido aumentar la complejidad del modelo**. Hacerlo solo incrementara la varianza sin reducir el error total.

In [ ]:
# --- Visualizacion del ruido irreducible (Error de Bayes) ---

np.random.seed(SEED)

n = 200
sigma_noise = 0.3
X = np.sort(np.random.uniform(0, 2 * np.pi, n))
noise = np.random.normal(0, sigma_noise, n)
y_true = np.sin(X)
y_observed = y_true + noise

# Prediccion del "modelo perfecto" (conoce f(x) = sin(x))
y_perfect_pred = np.sin(X)
residuals = y_observed - y_perfect_pred  # Estos son exactamente el ruido

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: funcion verdadera vs datos observados
ax1 = axes[0]
ax1.scatter(X, y_observed, color='gray', alpha=0.4, s=20, label='Datos observados')
ax1.plot(X, y_true, 'k-', linewidth=2, label='f(x) = sin(x) (funcion verdadera)')
ax1.fill_between(X, y_true - 2*sigma_noise, y_true + 2*sigma_noise,
                  alpha=0.15, color='red', label=f'Banda de ruido (+/- 2*sigma)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('El modelo perfecto aun tiene error residual')
ax1.legend(fontsize=9)

# Panel derecho: distribucion de los residuos
ax2 = axes[1]
ax2.hist(residuals, bins=25, density=True, alpha=0.7, color='#3498db',
         edgecolor='white', label='Residuos del modelo perfecto')
# Curva teorica
x_gauss = np.linspace(-1, 1, 100)
y_gauss = (1 / (sigma_noise * np.sqrt(2 * np.pi))) * np.exp(-x_gauss**2 / (2 * sigma_noise**2))
ax2.plot(x_gauss, y_gauss, 'r-', linewidth=2, label=f'N(0, {sigma_noise}^2) teorica')
ax2.axvline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('Residuo (y_obs - f(x))')
ax2.set_ylabel('Densidad')
ax2.set_title('Distribucion del ruido irreducible')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

mse_perfect = np.mean(residuals**2)
print(f"MSE del modelo perfecto (oraculo): {mse_perfect:.4f}")
print(f"Varianza teorica del ruido (sigma^2): {sigma_noise**2:.4f}")
print(f"Este es el piso de error que ningun modelo puede superar.")

## Aplicacion al proyecto C-MAPSS: techo de rendimiento

En el dataset C-MAPSS (prediccion de vida util remanente de motores), el concepto de **techo de rendimiento** es fundamental.

### Por que se aplica un cap en RUL = 130 ciclos

Cuando un motor esta lejos de la falla (por ejemplo, le quedan 200 ciclos), los sensores **no muestran degradacion significativa**. Los valores de los sensores son indistinguibles de un motor nuevo. Por lo tanto:

- **No hay senal en los datos** para distinguir RUL=200 de RUL=130. Intentar predecir esa diferencia es pedir al modelo que prediga **ruido puro**.
- El cap en 130 reconoce explicitamente este techo de rendimiento: "no pretendemos predecir lo que los datos no pueden decirnos".

### Sensores como sintomas, no como causas

Los 21 sensores del C-MAPSS son **sintomas observables** de la degradacion interna del motor. Pero existen **variables latentes** que afectan la vida util y que no estan capturadas:

- Microfracturas internas no detectables por sensores
- Variabilidad en la composicion del material
- Historial exacto de esfuerzos termicos y mecanicos

Estas variables latentes son la fuente del **ruido irreducible** en nuestro problema. Definen el piso de error que ningun modelo, por sofisticado que sea, puede superar con los sensores disponibles.

In [ ]:
# --- Descomposicion Bias^2 + Varianza + Ruido por complejidad ---

np.random.seed(SEED)

sigma_noise = 0.3
noise_var = sigma_noise ** 2
n_samples = 50
n_bootstraps = 100  # Numero de datasets bootstrap para estimar bias y varianza

X_fixed = np.linspace(0, 2 * np.pi, 80).reshape(-1, 1)
f_true = np.sin(X_fixed).ravel()

complexities = [1, 3, 5, 10, 15]
bias_sq_list = []
variance_list = []
noise_list = []

for deg in complexities:
    predictions = np.zeros((n_bootstraps, len(X_fixed)))
    
    for b in range(n_bootstraps):
        X_train = np.sort(np.random.uniform(0, 2 * np.pi, n_samples))
        y_train = np.sin(X_train) + np.random.normal(0, sigma_noise, n_samples)
        
        model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
        model.fit(X_train.reshape(-1, 1), y_train)
        predictions[b, :] = model.predict(X_fixed)
    
    # Bias^2: (E[prediccion] - f_true)^2 promediado sobre puntos x
    mean_pred = predictions.mean(axis=0)
    bias_sq = np.mean((mean_pred - f_true) ** 2)
    
    # Varianza: E[(prediccion - E[prediccion])^2] promediado sobre puntos x
    variance = np.mean(predictions.var(axis=0))
    
    bias_sq_list.append(bias_sq)
    variance_list.append(variance)
    noise_list.append(noise_var)

# Grafico de barras apiladas
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(complexities))
bar_width = 0.5

bars_noise = ax.bar(x_pos, noise_list, bar_width, label='Ruido irreducible',
                     color='#95a5a6', edgecolor='white')
bars_bias = ax.bar(x_pos, bias_sq_list, bar_width, bottom=noise_list,
                    label='Bias^2', color='#e74c3c', edgecolor='white')
bars_var = ax.bar(x_pos, variance_list, bar_width,
                   bottom=[n + b for n, b in zip(noise_list, bias_sq_list)],
                   label='Varianza', color='#3498db', edgecolor='white')

# Error total como linea
total_error = [n + b + v for n, b, v in zip(noise_list, bias_sq_list, variance_list)]
ax.plot(x_pos, total_error, 'ko-', linewidth=2, markersize=8, label='Error total', zorder=5)

ax.set_xlabel('Grado del polinomio (complejidad)')
ax.set_ylabel('Error (MSE)')
ax.set_title('Descomposicion del Error: Bias^2 + Varianza + Ruido')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(d) for d in complexities])
ax.legend(fontsize=11)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

# Tabla resumen
print(f"{'Grado':<8} {'Bias^2':<10} {'Varianza':<10} {'Ruido':<10} {'Total':<10}")
print("-" * 48)
for deg, b, v, n in zip(complexities, bias_sq_list, variance_list, noise_list):
    print(f"{deg:<8} {b:<10.4f} {v:<10.4f} {n:<10.4f} {b+v+n:<10.4f}")

## Resumen: estrategias segun la fuente de error

| Fuente de error | Como detectarla | Como reducirla |
|---|---|---|
| **Alto Bias** | Error alto en train Y en test. Las curvas de aprendizaje convergen a un valor alto. | Feature engineering, modelos mas complejos, menos regularizacion, transformaciones no lineales. |
| **Alta Varianza** | Error bajo en train pero alto en test (gap grande). | Mas datos de entrenamiento, regularizacion (L1/L2), dropout, pruning, ensemble methods, reducir features. |
| **Ruido Irreducible** | El error en test no baja mas sin importar la complejidad del modelo. | No se puede reducir con modelado. Solo con nuevas fuentes de datos o sensores adicionales. |

### Conceptos clave de este notebook

1. **Error = Bias^2 + Varianza + Ruido**: toda prediccion imperfecta se explica por estas tres fuentes.
2. **El tradeoff es inevitable**: reducir bias aumenta varianza y viceversa. El arte del ML es encontrar el equilibrio.
3. **El ruido define el techo**: no tiene sentido perseguir un error menor al ruido irreducible. Reconocer este limite ahorra tiempo y evita overfitting.
4. **Aplicacion practica**: el cap de RUL=130 en C-MAPSS es un ejemplo real de reconocer el techo de rendimiento impuesto por el ruido en los datos.